In [2]:
import sys
import os
import torch
import triton
# Add the parent directory to the Python path
sys.path.append(os.path.abspath('..'))
sys.path.append(os.path.abspath('C:\Program Files\Microsoft Visual Studio\18\Insiders\VC\Tools\MSVC\14.50.35717\bin\Hostx64\x64'))

import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import matplotlib.patches as patches
import torch
import copy
sys.path.append("/Users/LOCCO_Louise/Documents/Git/code_amaury")
from simu_PSF_polarMFM import *
from extract_experimental_psf import *
from tqdm import tqdm
import time
from torch.optim import SGD, Adam, AdamW
import torch.nn.functional as F

In [3]:
if torch.cuda.is_available():
    device = torch.device('cuda')
    print("Using GPU")
else:   
    device = torch.device('cpu')
    print("Using CPU")

Using GPU


In [4]:
d = np.array([.8,.8,.8])
d = np.array([np.mean(d)-0.350, np.mean(d), np.mean(d)+0.350])
QE = 0.92
EM = 200
sensitivity = 15.4

In [5]:
Nframe=50
raw = np.zeros((Nframe,6,214,129))

In [6]:
path_info = '/mnt/e/2026_01_19_SLB_1um/sample2/SLB1_10_NR40/SM/Calib_Polar_2026-01-19/images/RAW_DATA/image_Pos0.ome_results_fr1to7999_method=Propagation matrix_box-method=Fixed_box5.csv'
#path_info = 'E:\\2026_01_19_SLB_1um\\sample2\\SLB1_10_NR40\\SM\\Calib_Polar_2026-01-19\\images\\RAW_DATA\\image_Pos0.ome_results_fr1to7999_method=Propagation matrix_box-method=Fixed_box5.csv'

In [7]:
compute_M = torch.compile(compute_M)
PSF = torch.compile(PSF)

In [8]:
def extract_frames(frame_0, N_frame):
    error_indices = []
    for i in range(N_frame):
        number = str(frame_0 + i).zfill(4)
        print(number)
        path_data = '/mnt/e/2026_01_19_SLB_1um/sample2/SLB1_10_NR40/SM/Calib_Polar_2026-01-19/images/RAW_DATA/image_Pos0_reco/image_Pos0_'+number+'.tif'
        #path_data = 'E:\\2026_01_19_SLB_1um\\sample2\\SLB1_10_NR40\\SM\\Calib_Polar_2026-01-19\\images\\RAW_DATA\\image_Pos0_reco/image_Pos0_'+number+'.tif'
        raw_ = extract_raw(path_data)
        if raw_ is None:
            error_indices.append(i)
            continue
        else:
            raw[i] = raw_
        del(raw_)
    return raw, error_indices

def extract_positions(frame_0, N_frame, error_indices):
    index_frame = []
    x, y, z, rho, delta = [], [], [], [], []
    ind = 0
    for i in range(N_frame):
        if i not in error_indices:
            x__, y__, z__, rho__, delta__ = position_from_data(data, frame_0+i)
            x = np.concatenate((x, x__))
            y = np.concatenate((y, y__))
            z = np.concatenate((z, z__))
            rho = np.concatenate((rho, rho__))
            delta = np.concatenate((delta, delta__))
            for k in range(len(x__)):
                index_frame.append(ind)
        ind+=1
    index_frame=np.array(index_frame)
    return x, y, z, rho, delta, index_frame

def limit(x, lim, slope, upper=True):
    '''
    if upper:
       return torch.sum(torch.tensor(1/(1+torch.exp(-slope*(x-lim))), requires_grad=True, device=device))
    else:
        return torch.sum(torch.tensor(1/(1+torch.exp(slope*(x-lim))), requires_grad=True, device=device))
    '''
    if upper:
        return torch.sum(torch.exp((x-lim)*slope))
    else:
        return torch.sum(torch.exp(-1*(x-lim)*slope))
    
def loss_pos_torch(xp, yp, zp, rho, eta, delta, N_photons, data, second_plane, background, sigma, dim_simu, plot):
    u, v, M_ = compute_M(xp=xp, yp=yp, zp=zp, d=d_, x=xx, y=yy, th1=th1, phi=phi, Ex0=Ex0, Ex1=Ex1, Ex2=Ex2
                    , Ey0=Ey0, Ey1=Ey1, Ey2=Ey2, r=r, r_cut=r_cut, k=k_, f_o=f_o, phase_maskx=phase_mask, phase_masky=phase_mask, zernike_base=zernike_base, zernike_coefs_x=zernike_coefs_x, zernike_coefs_y=zernike_coefs_x,
                        second_plane=second_plane, polar_projections=polar_projections, N=N,
                    l_pixel=l_pixel, NA=NA, mag=mag, lambd=lambd, f_tube=f_tube, MAG=MAG, device=device, polar_offset=0., polar_offset2=0.)
    dim_data = 6
    h = PSF(rho=rho, eta=eta, delta=delta, M=M_, N_photons=N_photons)[:,:,:,dim_simu-dim_data:dim_simu+dim_data+1,dim_simu-dim_data:dim_simu+dim_data+1]
    loss = torch.sum(torch.pow(torch.sum(torch.add(h+torch.reshape(background, (h.shape[0],3,2))[:, :, :, None, None], -data), dim=(2,)), 2))
    x_bound = limit(xp, 5*0.12, 100, upper=True) + limit(xp, -5*0.12, 100, upper=False)
    y_bound = limit(yp, 5*0.12, 100, upper=True) + limit(yp, -5*0.12, 100, upper=False)
    z_bound = limit(zp, 5., 100, upper=True) + limit(zp, 0, 100, upper=False)
    return loss +x_bound+y_bound+z_bound
loss_pos = torch.compile(loss_pos_torch)

def loss_angle_with_M_torch(rho, eta, delta, N_photons, x_fine, y_fine, z_fine, zernx, zerny, data, background, sigma, dim_simu):#, plot):
    dim_data = 6
    u, v, M_ = compute_M(xp=x_fine, yp=y_fine, zp=z_fine, d=d_, x=xx, y=yy, th1=th1, phi=phi, Ex0=Ex0, Ex1=Ex1, Ex2=Ex2
                    , Ey0=Ey0, Ey1=Ey1, Ey2=Ey2, r=r, r_cut=r_cut, k=k_, f_o=f_o, phase_masky=phase_mask, phase_maskx=phase_mask, zernike_base=zernike_base, zernike_coefs_x=torch.reshape(zernx, (3,15)), zernike_coefs_y=torch.reshape(zerny, (3,15)),
                        second_plane=second_plane, polar_projections=polar_projections, N=N,
                    l_pixel=l_pixel, NA=NA, mag=mag, lambd=lambd, f_tube=f_tube, MAG=MAG, device=device)
    h = PSF(rho=rho, eta=eta, delta=delta, M=M_, N_photons=N_photons)[:,:,:,dim_simu-dim_data:dim_simu+dim_data+1,dim_simu-dim_data:dim_simu+dim_data+1]
    
    loss = torch.sum(torch.add(h, -(data+sigma**2)*torch.log(h+torch.reshape(background, (h.shape[0],3,2))[:, :, :, None, None]+sigma**2)))
    delta_bound = limit(delta, 180, 100, upper=True) + limit(delta, 1, 100, upper=False)
    '''if plot:
        for nb in range(data.shape[0]):
            print(rho, eta, delta)
            maxi = max(np.max(data[nb,:,:].flatten().cpu().detach().numpy()), np.max(h[nb,:,:].flatten().cpu().detach().numpy()))
            fig, ax = plt.subplots(3,2)
            ax[0,0].imshow(data[nb,0,0].cpu().detach().numpy(), vmin=0., vmax=maxi, cmap='gray')
            ax[0,0].scatter(x_fine[nb].cpu().detach().numpy()/0.120+6, y_fine[nb].cpu().detach().numpy()/0.120+6, s=10, c='r', marker='x')
            ax[1,0].imshow(data[nb,1,0].cpu().detach().numpy() , vmin=0., vmax=maxi, cmap='gray')
            ax[1,0].scatter(x_fine[nb].cpu().detach().numpy()/0.120+6, y_fine[nb].cpu().detach().numpy()/0.120+6, s=10, c='r', marker='x')
            ax[2,0].imshow(data[nb,2,0].cpu().detach().numpy(), vmin=0., vmax=maxi, cmap='gray')
            ax[2,0].scatter(x_fine[nb].cpu().detach().numpy()/0.120+6, y_fine[nb].cpu().detach().numpy()/0.120+6, s=10, c='r', marker='x')
            ax[0,1].imshow(data[nb,0,1].cpu().detach().numpy(), vmin=0., vmax=maxi, cmap='gray')
            ax[0,1].scatter(x_fine[nb].cpu().detach().numpy()/0.120+6, y_fine[nb].cpu().detach().numpy()/0.120+6, s=10, c='r', marker='x')
            ax[1,1].imshow(data[nb,1,1].cpu().detach().numpy(), vmin=0., vmax=maxi, cmap='gray')
            ax[1,1].scatter(x_fine[nb].cpu().detach().numpy()/0.120+6, y_fine[nb].cpu().detach().numpy()/0.120+6, s=10, c='r', marker='x')
            ax[2,1].imshow(data[nb,2,1].cpu().detach().numpy(), vmin=0., vmax=maxi, cmap='gray')
            ax[2,1].scatter(x_fine[nb].cpu().detach().numpy()/0.120+6, y_fine[nb].cpu().detach().numpy()/0.120+6, s=10, c='r', marker='x')
            plt.show()
            del(fig, ax)
            fig, ax = plt.subplots(3,2)
            ax[0,0].imshow(h[nb,0,0].cpu().detach().numpy(), vmin=0., vmax=maxi, cmap='gray')
            ax[0,0].scatter(x_fine[nb].cpu().detach().numpy()/0.120+6, y_fine[nb].cpu().detach().numpy()/0.120+6, s=10, c='r', marker='x')
            ax[1,0].imshow(h[nb,1,0].cpu().detach().numpy(), vmin=0., vmax=maxi, cmap='gray')
            ax[1,0].scatter(x_fine[nb].cpu().detach().numpy()/0.120+6, y_fine[nb].cpu().detach().numpy()/0.120+6, s=10, c='r', marker='x')
            ax[2,0].imshow(h[nb,2,0].cpu().detach().numpy(), vmin=0., vmax=maxi, cmap='gray')
            ax[2,0].scatter(x_fine[nb].cpu().detach().numpy()/0.120+6, y_fine[nb].cpu().detach().numpy()/0.120+6, s=10, c='r', marker='x')
            ax[0,1].imshow(h[nb,0,1].cpu().detach().numpy(), vmin=0., vmax=maxi, cmap='gray')
            ax[0,1].scatter(x_fine[nb].cpu().detach().numpy()/0.120+6, y_fine[nb].cpu().detach().numpy()/0.120+6, s=10, c='r', marker='x')
            ax[1,1].imshow(h[nb,1,1].cpu().detach().numpy(), vmin=0., vmax=maxi, cmap='gray')
            ax[1,1].scatter(x_fine[nb].cpu().detach().numpy()/0.120+6, y_fine[nb].cpu().detach().numpy()/0.120+6, s=10, c='r', marker='x')
            ax[2,1].imshow(h[nb,2,1].cpu().detach().numpy(), vmin=0., vmax=maxi, cmap='gray')
            ax[2,1].scatter(x_fine[nb].cpu().detach().numpy()/0.120+6, y_fine[nb].cpu().detach().numpy()/0.120+6, s=10, c='r', marker='x')
            plt.show()
            del(fig, ax)'''
    return loss + 1000.*(delta_bound) #+ N_bound #+ 100000*torch.sum(h**2)
loss_angle_with_M = torch.compile(loss_angle_with_M_torch)

def score_eval(M_, rho, eta, delta, N_photons, data, background, sigma, dim_simu):
    dim_data = 6
    h = PSF(rho=rho, eta=eta, delta=delta, M=M_, N_photons=N_photons)[:,:,:,dim_simu-dim_data:dim_simu+dim_data+1,dim_simu-dim_data:dim_simu+dim_data+1]
    score = torch.sum(torch.add(h, -(data+sigma**2)*torch.log(h+background+sigma**2)), dim=(1,2,3,4))
    return score.numpy() 

In [9]:
data = pos_from_csv(path_info)

In [10]:
N_batch = 500
batch_offset = 0

In [11]:
# polar calibration
'''J_dichroic = np.array([[0.2646684         ,            -0.08827579 + 1j*0.15587579],[
      0.15437593 + 1j*0.084718674   ,  0.23629372 - 1j*0.13453841]]) 

J_dichroic = np.array([[0.7838338      ,               -0.25981125 + 1j*  -0.48329058],[
      -0.4230177 + 1j*  0.27765664  ,   -0.7788276 + 1j*  -0.28660256
]]) # this one is for the abstract

J_dichroic = np.array([[0.6152142     ,               -0.07071821 + 1j*  -0.32384807],[
      -0.11067753 + 1j*  0.28322944   ,   -0.6181744 + 1j*  -0.08643947
]])'''
J_dichroic = np.array([[0.8717185                 ,    -0.097953975 + 1j*  -0.45857465],[
      -0.16148566 + 1j*   0.39992934   ,  -0.87400675 + 1j*  -0.13822438
                        ]])
'''
# version for -45 angle
J_dichroic = np.array([[0.5977892      ,               -0.067152604 + 1j*  -0.3144654],[
      0.11054159 + 1j* -0.27431256  ,   0.59944284 +1j*  0.09444063]])'''

'\n# version for -45 angle\nJ_dichroic = np.array([[0.5977892      ,               -0.067152604 + 1j*  -0.3144654],[\n      0.11054159 + 1j* -0.27431256  ,   0.59944284 +1j*  0.09444063]])'

In [12]:
np.sum(np.real(J_dichroic)**2+np.imag(J_dichroic)**2)

1.9487937077335913

In [13]:
torch.cuda.empty_cache()
torch.cuda.ipc_collect()

In [14]:
for batch_number in range(N_batch):
    t0 = time.time()
    # extracteing the raw 6-stack tiff files
    raw, error_indices = extract_frames((batch_number+batch_offset)*Nframe+1, Nframe)
    # extracting the position from Louise pipeline
    x, y, z, rho, delta, index_frame = extract_positions((batch_number+batch_offset)*Nframe+1, Nframe, error_indices)
    # converting to photon count
    raw = raw*sensitivity/(QE*EM)
    # these quantites are used to evaluated the noise and inserted into the loss
    sigma = np.std(raw.flatten())
    background = np.mean(raw.flatten())

    # removing all the PSF where a parameter is evaluated to nan in Louise pipeline
    nb = len(x)
    for k, ele in enumerate(x):
        if np.isnan(x[nb-1-k]) or np.isnan(y[nb-1-k]) or np.isnan(z[nb-1-k]) or np.isnan(rho[nb-1-k]) or np.isnan(delta[nb-1-k]):
            x = np.delete(x,nb-1-k,0)
            y = np.delete(y,nb-1-k,0)
            z = np.delete(z,nb-1-k,0)
            rho = np.delete(rho,nb-1-k,0)
            delta = np.delete(delta,nb-1-k,0)
            index_frame = np.delete(index_frame,nb-1-k,0)
            
    # extracting the psf from the files
    single_psf = extract_raw_xy(raw[0], x[index_frame==0], y[index_frame==0])
    for i in range(1,Nframe):
        single_psf = np.concatenate((single_psf, extract_raw_xy(raw[i], x[index_frame==i], y[index_frame==i])))

    # dimenstion matching to have x in horizontal and y in vertical when considering what appears in a tiff file
    #single_psf = single_psf[:,::-1]
    #single_psf = np.transpose(single_psf[:,:,:,:,:], axes=(0,1,2,4,3))
    single_psf = single_psf[:,:,:,::-1,:]
    #x = np.max(x)-x
    x, y = y, x

    # nb of photons by plane roughly evaluated
    Nstart_by_plane = copy.deepcopy(np.sum(single_psf, axis=(2,3,4)) - background*len(single_psf[0,0].flatten()))

    # removing all the PSF where there are two emitters, either too bright or the middle plane less bright than the extremal ones
    nb = len(x)
    
    for k, ele in enumerate(x):
        if ((Nstart_by_plane[k,0]>Nstart_by_plane[k,1]) & (Nstart_by_plane[k,2]>Nstart_by_plane[k,1])) | (Nstart_by_plane[k,0]+Nstart_by_plane[k,1]+Nstart_by_plane[k,2]<2000):
            x = np.delete(x,nb-1-k,0)
            y = np.delete(y,nb-1-k,0)
            z = np.delete(z,nb-1-k,0)
            rho = np.delete(rho,nb-1-k,0)
            delta = np.delete(delta,nb-1-k,0)
            index_frame = np.delete(index_frame,nb-1-k,0)
            single_psf = np.delete(single_psf,nb-1-k,0)
    
    NPSF = len(x)

    # strating parameters (could be a first evaluation with coarse algo)
    x_start = torch.tensor([0. for k in range(len(x))], requires_grad=False, device=device)
    y_start = torch.tensor([0. for k in range(len(x))], requires_grad=False, device=device)
    z_exp =  torch.tensor([0.6 for k in range(len(x))], requires_grad=False, device=device) 

    # microscope parameters
    d_ = -torch.tensor([d[1] for k in range(len(x))], requires_grad=False, device=device)
    second_plane = torch.tensor([d[1]-d[0], 0, d[1]-d[2]], device=device)
    polar_projections = np.array([0, 45, 0])

    N=torch.tensor(80, device=device)
    l_pixel=torch.tensor(16, device=device)
    NA=torch.tensor(1.4, device=device)
    mag=torch.tensor(100, device=device)
    lambd=torch.tensor(638, device=device)
    f_tube=torch.tensor(200, device=device)
    MAG=torch.tensor(200/150, device=device)
    xx, yy, th1, phi, [Ex0, Ex1, Ex2], [Ey0, Ey1, Ey2], r, r_cut, k_, f_o = vectorial_BFP_perfect_focus(N, NA=NA, mag=mag, lambd=lambd, f_tube=f_tube, device=device, J_dichroic=J_dichroic)

    if batch_number==0:
        phase_mask = torch.stack([torch.ones((N,N), device=device), torch.ones((N,N), device=device), torch.ones((N,N), device=device)])
        zernike_base = generate_zernike_base(r_cut=r_cut, N=N, zernike_order=4, device=device)
        zernike_coefs_x = torch.zeros((3,15), device=device)
        zernike_coefs_y = torch.zeros((3,15), device=device)

    # convert to tensor
    noisy_psf = torch.tensor([single_psf[k] for k in range(len(x))], device=device, dtype=torch.float32)

    # starting point, could do a rough estimation first
    rho_start = torch.tensor(rho-60, device=device)
    eta_start = torch.tensor([90. for k in range(NPSF)], requires_grad=False, device=device)
    delta_start = torch.tensor(delta, device=device)

    # gradient descent parameters
    Nstart = torch.tensor([5000. for g in range(NPSF)], device=device)#
    #Nstart = copy.deepcopy(torch.sum(noisy_psf, dim=(1,2,3,4)) - background*len(noisy_psf[0].flatten()))
    
    u, v, M = compute_M(xp=x_start, yp=y_start, zp=z_exp, d=d_, x=xx, y=yy, th1=th1, phi=phi, Ex0=Ex0, Ex1=Ex1, Ex2=Ex2
                    , Ey0=Ey0, Ey1=Ey1, Ey2=Ey2, r=r, r_cut=r_cut, k=k_, f_o=f_o, phase_maskx=phase_mask, phase_masky=phase_mask, zernike_base=zernike_base, zernike_coefs_x=zernike_coefs_x, zernike_coefs_y=zernike_coefs_y,
                        second_plane=second_plane, polar_projections=polar_projections, N=N, l_pixel=l_pixel
                    , NA=NA, mag=mag, lambd=lambd, f_tube=f_tube, MAG=MAG, device=device, polar_offset=0., polar_offset2=0.)
    h = PSF(rho=rho_start, eta=eta_start, delta=delta_start, M=M, N_photons=Nstart)
    
    dim_simu = int(h.shape[-1]//2)

    background_array = torch.tensor(background*np.ones((NPSF,3,2)) ,device=device)
    
    params = torch.cat((x_start, y_start, z_exp, Nstart/3000, background_array.flatten()))
    params.requires_grad=True

    angle_rd = torch.tensor([180. for k in range(NPSF)], requires_grad=False, device=device)
    optimizer = torch.optim.Adam([params], lr=0.05)
    num_epochs_max = 80
    loss_ = []
    z__ = []
    N__ = []
    x__ =[]
    for i in tqdm(range(num_epochs_max)):
        optimizer.zero_grad()  # Reset gradients
        loss = loss_pos(params[0:NPSF], params[NPSF:2*NPSF], params[2*NPSF:3*NPSF], angle_rd
                            , angle_rd, angle_rd, params[3*NPSF:4*NPSF]*3000, noisy_psf, second_plane, params[4*NPSF:10*NPSF], sigma, dim_simu, plot=False) 
        loss_.append(loss.cpu().detach().numpy())
        z__.append(params[2*NPSF:3*NPSF].cpu().detach().numpy())
        N__.append((params[3*NPSF:4*NPSF]*3000).cpu().detach().numpy())
        x__.append((params[0*NPSF:1*NPSF]*3000).cpu().detach().numpy())
        loss.backward()  # Backpropagation
        optimizer.step()  # Update parameters
    ax = plt.plot(loss_)
    plt.ylim((np.min(np.array(loss_)), np.max(np.array(loss_))))
    plt.show()
    ax = plt.plot(z__)
    plt.show()
    ax = plt.plot(N__)
    plt.show()
    ax = plt.plot(x__)
    plt.ylim((np.min(np.array(x__)), np.max(np.array(x__))))
    plt.show()
    del(ax, loss_, z__, N__)

    x_found = params[0:NPSF].detach()
    y_found = params[NPSF:2*NPSF].detach()
    z_found = params[2*NPSF:3*NPSF].detach()
    N_found = params[3*NPSF:4*NPSF].detach()*3000
    bacground_array_found = params[4*NPSF:10*NPSF].detach()
    del(params, loss)
    print('NPSF = ', NPSF)

#########################################################################################################################

    zern_x = torch.tensor(np.zeros(3*15), device=device)
    zern_y = torch.tensor(np.zeros(3*15), device=device)

    params = torch.cat((rho_start, eta_start, delta_start, N_found))#, x_found*10, y_found*10, z_found))
    params.requires_grad=True

    # Use Stochastic Gradient Descent (SGD) to optimize params
    optimizer = torch.optim.Adam([params], lr=0.7)  # Learning rate = 0.01

    num_epochs_max = 200
    loss_ = []
    eta_ = []
    rho_ = []
    delta_ = []
    for i in tqdm(range(num_epochs_max)):
        optimizer.zero_grad()  # Reset gradients
        loss = loss_angle_with_M(params[:NPSF], params[1*NPSF:2*NPSF], params[2*NPSF:3*NPSF], params[3*NPSF:4*NPSF], x_found, y_found, z_found, zern_x, zern_y, noisy_psf, bacground_array_found, sigma, dim_simu)#, plot=(i==num_epochs_max-1))
        loss_.append(loss.cpu().detach().numpy())
        eta_.append(params[1*NPSF:2*NPSF].cpu().detach().numpy())
        rho_.append(params[0*NPSF:1*NPSF].cpu().detach().numpy())
        delta_.append(params[2*NPSF:3*NPSF].cpu().detach().numpy())
        loss.backward()  # Backpropagation
        optimizer.step()  # Update parameters
    fig, ax = plt.subplots(2,2)
    ax[0,0].plot(loss_) 
    ax[0,0].set_ylim((np.min(np.array(loss_)), np.max(np.array(loss_))))
    ax[0,1].plot(eta_)
    ax[1,0].plot(rho_)
    ax[1,1].plot(delta_)
    plt.show()
    del(fig, ax)

    rho_found=params[0:NPSF].detach()%180
    eta_found=params[1*NPSF:2*NPSF].detach()%180
    delta_found=params[2*NPSF:3*NPSF].detach()
    N_found2 = params[3*NPSF:4*NPSF].detach()
    #x_found = params[4*NPSF:5*NPSF].detach()/10
    #y_found = params[5*NPSF:6*NPSF].detach()/10
    #z_found = params[6*NPSF:7*NPSF].detach()
    zernx = torch.reshape(zern_x, (3,15)).detach()
    zerny = torch.reshape(zern_x, (3,15)).detach()
    del(params, loss, eta_, loss_)
    u, v, M = compute_M(xp=x_found, yp=y_found, zp=z_found, d=d_, x=xx, y=yy, th1=th1, phi=phi, Ex0=Ex0, Ex1=Ex1, Ex2=Ex2
                    , Ey0=Ey0, Ey1=Ey1, Ey2=Ey2, r=r, r_cut=r_cut, k=k_, f_o=f_o, phase_maskx=phase_mask, phase_masky=phase_mask, zernike_base=zernike_base, zernike_coefs_x=zernike_coefs_x, zernike_coefs_y=zernike_coefs_x,
                        second_plane=second_plane, polar_projections=polar_projections, N=N, l_pixel=l_pixel
                    , NA=NA, mag=mag, lambd=lambd, f_tube=f_tube, MAG=MAG, device=device)
    score = score_eval(M.detach().cpu(), rho_found.cpu(), eta_found.cpu(), delta_found.cpu(), N_found2.cpu(), noisy_psf.cpu(), background, sigma, dim_simu)
    x_ = (x/0.120).astype(int)*0.12 + x_found.cpu().detach().numpy()
    y_ = (y/0.120).astype(int)*0.12 + y_found.cpu().detach().numpy()
    np.savez_compressed('Bureau\\2026_01_19_I\\'+str(int(batch_number)+1+batch_offset)+'.npz', frame = index_frame, x=x_, y=y_, z=1000*z_found.cpu().detach().numpy(), N_photons=N_found2.cpu().detach().numpy(), offset_proj=np.nan, offset_proj2=np.nan, rho=rho_found.cpu().detach().numpy(), eta=eta_found.cpu().detach().numpy(), delta=delta_found.cpu().detach().numpy(), score=score, x_start=x, y_start=y, z_start=z, rho_start=rho, delta_start=delta, zernx=zernx.cpu().detach().numpy(), zerny=zerny.cpu().detach().numpy())


0001
0002
0003
0004
0005
0006
0007
0008
0009
0010
0011
0012
0013
0014
0015
0016
0017
0018
0019
0020
0021
0022
0023
0024
0025
0026
0027
0028
0029
0030
0031
0032
0033
0034
0035
0036
0037
0038
0039
0040
0041
0042
0043
0044
0045
0046
0047
0048
0049
0050


/tmp/ipykernel_1612/3056911682.py:80: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /pytorch/torch/csrc/utils/tensor_new.cpp:253.)
  noisy_psf = torch.tensor([single_psf[k] for k in range(len(x))], device=device, dtype=torch.float32)
/home/amaury/miniconda3/envs/torch_linux/lib/python3.11/site-packages/torch/_inductor/lowering.py:2207: UserWarning: Torchinductor does not support code generation for complex operators. Performance may be worse than eager.
  warnings.warn(
/home/amaury/miniconda3/envs/torch_linux/lib/python3.11/site-packages/torch/functional.py:505: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at /pytorch/aten/src/ATen/native/TensorShape.cpp:4381.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]
W0123 

InductorError: AttributeError: 'complex' object has no attribute 'get_name'